# 📊 The Confusion Matrix

Welcome to the hands-on explanation notebook for the **Confusion Matrix**! In this notebook, we will:
1. Generate predictions for a 3-class classification problem (`flange`, `valve`, `gauge`) where some class confusion occurs.
2. Implement both the **Raw** and **Normalized** Confusion Matrices from scratch using NumPy.
3. Verify our scratch implementation against `scikit-learn`.
4. Plot the matrices using heatmaps to visually isolate where the model is struggling.
5. Explain the YOLO multi-class confusion matrix, including how it handles the **Background** class.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Set seed for reproducibility
np.random.seed(42)

## 1. Data Generation

We simulate labels and predictions for 150 components:
*   Class 0: `flange`
*   Class 1: `valve`
*   Class 2: `gauge`

The model is excellent at classifying flanges, but frequently misclassifies gauges as valves.

In [ ]:
# Generate true classes: equal proportions (50 of each)
y_true = np.concatenate([np.zeros(50), np.ones(50), np.ones(50)*2]).astype(int)

# Generate predictions with typical confusion
y_pred = y_true.copy()

# Flanges (Class 0): 90% correct, 10% confused as valves
confuse_0 = np.random.choice(50, 5, replace=False)
y_pred[confuse_0] = 1

# Valves (Class 1): 80% correct, 10% confused as flanges, 10% as gauges
confuse_1_to_0 = np.random.choice(50, 5, replace=False) + 50
confuse_1_to_2 = np.random.choice(50, 5, replace=False) + 50
y_pred[confuse_1_to_0] = 0
y_pred[confuse_1_to_2] = 2

# Gauges (Class 2): 60% correct, 30% confused as valves, 10% as flanges
confuse_2_to_1 = np.random.choice(50, 15, replace=False) + 100
confuse_2_to_0 = np.random.choice(50, 5, replace=False) + 100
y_pred[confuse_2_to_1] = 1
y_pred[confuse_2_to_0] = 0

## 2. Calculating the Confusion Matrix from Scratch

Let's write a function to construct the $3\times3$ confusion matrix:
-   **Rows:** Represent the actual class.
-   **Columns:** Represent the predicted class.
-   Element $C_{i,j}$ counts how many times a sample of true class $i$ was predicted as class $j$.

In [ ]:
def custom_confusion_matrix(y_true, y_pred, n_classes=3):
    """
    Construct a raw confusion matrix from scratch.
    """
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def custom_normalized_confusion_matrix(cm):
    """
    Normalize confusion matrix by row sums (actual classes).
    """
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return cm.astype(float) / row_sums

# Run scratch functions
cm_raw = custom_confusion_matrix(y_true, y_pred, n_classes=3)
cm_norm = custom_normalized_confusion_matrix(cm_raw)

# Get sklearn version for comparison
cm_sklearn = confusion_matrix(y_true, y_pred)

print("--- Custom Raw Matrix ---")
print(cm_raw)
print("\n--- Sklearn Raw Matrix ---")
print(cm_sklearn)
print("\nVerify identical matches:", np.array_equal(cm_raw, cm_sklearn))

print("\n--- Custom Normalized Matrix ---")
print(np.round(cm_norm, 4))

## 3. Visualizing the Confusion Matrix

Let's plot both side-by-side using heatmaps.

In [ ]:
class_names = ['flange', 'valve', 'gauge']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Raw Matrix
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Raw Confusion Matrix")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("Actual Class")

# Plot Normalized Matrix
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted Class")
axes[1].set_ylabel("Actual Class")

plt.tight_layout()
plt.show()

Observe that the diagonal of the normalized matrix shows the class-specific recall accuracies:
*   Flange class accuracy: 90%
*   Valve class accuracy: 80%
*   Gauge class accuracy: 60%
The cell at row `gauge` and column `valve` shows `0.30`, proving that the model is confusing 30% of gauges as valves.

## 💡 Connection to Computer Vision & YOLO
In object detection, how does YOLO evaluate a confusion matrix since there are no fixed "samples" (unlike simple image classification)?
YOLO extends the confusion matrix by adding a **Background** row and column:
1.  **Prediction Mismatch:** If a ground truth bounding box of `control-valve` has no matching predicted box with $IoU \ge 0.45$, YOLO flags it as a **missed object**. This is registered in the matrix at row `control-valve`, column `Background` (a False Negative).
2.  **False Alarm:** If the model predicts a bounding box with class label `flange` over an area of the image that does not overlap with any ground truth box, it is counted as a **false alarm**. This is registered in the matrix at row `Background`, column `flange` (a False Positive).
This allows YOLO to evaluate classification confusions alongside localization errors (missing objects or predicting background noise).